In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["stay_id"]
)

inputevents_df = pd.read_parquet(PROCESSED_DATA_DIR / "inputevents_clean.parquet")

print(icustays_df.shape)
print(inputevents_df.shape)
print(inputevents_df["input_group"].value_counts())

(94444, 1)
(10918301, 30)
input_group
crystalloid             2535692
other_fluid             1965822
oral_gastric            1269936
sedative_analgesic      1219646
other_medication        1145827
vasopressor_inotrope     849098
antibiotic               758585
insulin                  474814
anticoagulant            258102
enteral_nutrition        255461
blood_product            180511
other                      4807
Name: count, dtype: int64


In [3]:
feature_window_hours = 24

inputevents_24h_df = inputevents_df[
    inputevents_df["hours_from_icu_admit_start"].le(feature_window_hours)
    & inputevents_df["hours_from_icu_admit_end"].ge(0)
].copy()

inputevents_24h_df["window_overlap_start_hour"] = inputevents_24h_df["hours_from_icu_admit_start"].clip(lower=0)
inputevents_24h_df["window_overlap_end_hour"] = inputevents_24h_df["hours_from_icu_admit_end"].clip(upper=feature_window_hours)
inputevents_24h_df["window_overlap_duration_hours"] = (
    inputevents_24h_df["window_overlap_end_hour"] - inputevents_24h_df["window_overlap_start_hour"]
).clip(lower=0)

inputevents_24h_df["window_overlap_fraction"] = 0.0
positive_duration_mask = inputevents_24h_df["duration_hours"].gt(0)
inputevents_24h_df.loc[positive_duration_mask, "window_overlap_fraction"] = (
    inputevents_24h_df.loc[positive_duration_mask, "window_overlap_duration_hours"]
    / inputevents_24h_df.loc[positive_duration_mask, "duration_hours"]
).clip(0, 1)

bolus_mask = ~positive_duration_mask & inputevents_24h_df["hours_from_icu_admit_start"].between(0, feature_window_hours, inclusive="both")
inputevents_24h_df.loc[bolus_mask, "window_overlap_fraction"] = 1.0

inputevents_24h_df["amount_24h"] = inputevents_24h_df["amount"] * inputevents_24h_df["window_overlap_fraction"]

print(inputevents_24h_df.shape)
print(inputevents_24h_df["input_group"].value_counts())

(3206226, 35)
input_group
crystalloid             841283
other_fluid             666379
sedative_analgesic      384461
other_medication        321724
vasopressor_inotrope    291501
oral_gastric            210832
antibiotic              176929
insulin                 157410
blood_product            89517
anticoagulant            55694
enteral_nutrition        10360
other                      136
Name: count, dtype: int64


In [4]:
base_features_df = (
    inputevents_24h_df
    .groupby("stay_id")
    .agg(
        input_event_count_24h=("itemid", "size"),
        unique_input_item_count_24h=("itemid", "nunique"),
        total_input_duration_hours_24h=("window_overlap_duration_hours", "sum"),
    )
    .reset_index()
)

base_features_df.head()

,stay_id,input_event_count_24h,unique_input_item_count_24h,total_input_duration_hours_24h
0,30000153,36,12,45.950000
1,30000213,25,8,120.250000
2,30000484,29,12,39.410000
3,30000646,36,7,79.389444
4,30001148,30,12,25.416667


In [5]:
group_features_df = (
    inputevents_24h_df
    .groupby(["stay_id", "input_group"])
    .agg(
        event_count=("itemid", "size"),
        duration_hours=("window_overlap_duration_hours", "sum"),
    )
    .reset_index()
)

group_features_wide_df = group_features_df.pivot(
    index="stay_id",
    columns="input_group"
)

group_features_wide_df.columns = [
    f"{input_group}_{stat}_24h"
    for stat, input_group in group_features_wide_df.columns
]

group_features_wide_df = group_features_wide_df.reset_index()

group_features_wide_df.head()

,stay_id,antibiotic_event_count_24h,anticoagulant_event_count_24h,blood_product_event_count_24h,crystalloid_event_count_24h,enteral_nutrition_event_count_24h,insulin_event_count_24h,oral_gastric_event_count_24h,other_event_count_24h,other_fluid_event_count_24h,...,blood_product_duration_hours_24h,crystalloid_duration_hours_24h,enteral_nutrition_duration_hours_24h,insulin_duration_hours_24h,oral_gastric_duration_hours_24h,other_duration_hours_24h,other_fluid_duration_hours_24h,other_medication_duration_hours_24h,sedative_analgesic_duration_hours_24h,vasopressor_inotrope_duration_hours_24h
0,30000153,NaN,NaN,NaN,10.0,NaN,1.0,NaN,NaN,6.0,...,NaN,39.216667,NaN,0.016667,NaN,NaN,3.250000,0.100000,3.366667,NaN
1,30000213,NaN,NaN,NaN,2.0,1.0,4.0,1.0,NaN,8.0,...,NaN,24.633333,9.850000,1.666667,0.016667,NaN,42.833333,0.033333,41.216667,NaN
2,30000484,4.0,3.0,2.0,6.0,5.0,NaN,1.0,NaN,5.0,...,2.666667,10.375556,15.825556,NaN,0.016667,NaN,0.083333,NaN,NaN,10.325556
3,30000646,5.0,NaN,NaN,8.0,NaN,NaN,7.0,NaN,11.0,...,NaN,59.562222,NaN,NaN,0.116667,NaN,9.860556,NaN,NaN,9.766667
4,30001148,2.0,NaN,NaN,5.0,NaN,2.0,3.0,NaN,6.0,...,NaN,12.033333,NaN,2.766667,0.050000,NaN,6.583333,0.033333,1.950000,1.966667


In [6]:
exposure_features_df = (
    inputevents_24h_df
    .groupby(["stay_id", "input_group"])
    .size()
    .unstack(fill_value=0)
    .gt(0)
    .astype(int)
)

exposure_features_df.columns = [
    f"{input_group}_used_24h"
    for input_group in exposure_features_df.columns
]

exposure_features_df = exposure_features_df.reset_index()

exposure_features_df.head()

,stay_id,antibiotic_used_24h,anticoagulant_used_24h,blood_product_used_24h,crystalloid_used_24h,enteral_nutrition_used_24h,insulin_used_24h,oral_gastric_used_24h,other_used_24h,other_fluid_used_24h,other_medication_used_24h,sedative_analgesic_used_24h,vasopressor_inotrope_used_24h
0,30000153,0,0,0,1,0,1,0,0,1,1,1,0
1,30000213,0,0,0,1,1,1,1,0,1,1,1,0
2,30000484,1,1,1,1,1,0,1,0,1,0,0,1
3,30000646,1,0,0,1,0,0,1,0,1,0,0,1
4,30001148,1,0,0,1,0,1,1,0,1,1,1,1


In [7]:
amount_features_df = (
    inputevents_24h_df
    .groupby(["stay_id", "input_group", "amountuom_clean"])["amount_24h"]
    .sum()
    .reset_index()
)

amount_features_df["feature_name"] = (
    amount_features_df["input_group"]
    + "_"
    + amount_features_df["amountuom_clean"]
    + "_total_24h"
)

amount_features_wide_df = amount_features_df.pivot(
    index="stay_id",
    columns="feature_name",
    values="amount_24h"
).reset_index()

amount_features_wide_df.head()

feature_name,stay_id,antibiotic_dose_total_24h,antibiotic_grams_total_24h,antibiotic_mcg_total_24h,antibiotic_mg_total_24h,antibiotic_ml_total_24h,anticoagulant_dose_total_24h,anticoagulant_mg_total_24h,anticoagulant_units_total_24h,blood_product_dose_total_24h,...,other_medication_mcg_total_24h,other_medication_meq_total_24h,other_medication_mg_total_24h,other_ml_total_24h,other_ounces_total_24h,sedative_analgesic_mcg_total_24h,sedative_analgesic_mg_total_24h,vasopressor_inotrope_mcg_total_24h,vasopressor_inotrope_mg_total_24h,vasopressor_inotrope_units_total_24h
0,30000153,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.000000,NaN,NaN,224.999995,521.590154,NaN,NaN,NaN
1,30000213,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,239.999998,NaN,NaN,NaN,1114.879771,NaN,NaN,NaN
2,30000484,4.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178.817283,NaN
3,30000646,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.796645,NaN
4,30001148,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,32.500000,NaN,NaN,NaN,322.009155,NaN,3.443385,NaN


In [8]:
inputevents_features_df = icustays_df.merge(
    base_features_df,
    on="stay_id",
    how="left"
)

for feature_df in [group_features_wide_df, exposure_features_df, amount_features_wide_df]:
    inputevents_features_df = inputevents_features_df.merge(
        feature_df,
        on="stay_id",
        how="left"
    )

count_cols = [col for col in inputevents_features_df.columns if col.endswith("_count_24h")]
used_cols = [col for col in inputevents_features_df.columns if col.endswith("_used_24h")]
duration_cols = [col for col in inputevents_features_df.columns if col.endswith("_duration_hours_24h")]
amount_cols = [col for col in inputevents_features_df.columns if col.endswith("_total_24h")]

fill_zero_cols = count_cols + used_cols + duration_cols + amount_cols
inputevents_features_df[fill_zero_cols] = inputevents_features_df[fill_zero_cols].fillna(0)
inputevents_features_df[used_cols] = inputevents_features_df[used_cols].astype(int)

print(inputevents_features_df.shape)
print(inputevents_features_df["stay_id"].duplicated().sum())
print(inputevents_features_df[used_cols].sum().sort_values(ascending=False))

(94444, 89)
0
crystalloid_used_24h             78395
other_fluid_used_24h             62719
other_medication_used_24h        59454
oral_gastric_used_24h            58247
antibiotic_used_24h              46000
sedative_analgesic_used_24h      37504
insulin_used_24h                 29094
anticoagulant_used_24h           26308
vasopressor_inotrope_used_24h    25819
blood_product_used_24h           25567
enteral_nutrition_used_24h        4847
other_used_24h                     101
dtype: int64


In [9]:
inputevents_features_df.to_parquet(
    PROCESSED_DATA_DIR / "inputevents_features.parquet",
    index=False
)